In [ ]:
# ── 1. IMPORTS ────────────────────────────────────────────────────────────────
import os
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm  # Use notebook version for cleaner progress bars

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc,
    accuracy_score
)
from sklearn.preprocessing import label_binarize

# ── 2. GLOBAL PLOT STYLE  (Optimized for Professional Reports) ───────────────
PALETTE   = ["#E74C3C", "#2980B9", "#27AE60"]   # Crimson / Cobalt / Emerald
BG_COLOR  = "#FFFFFF"                           # Crisp white for documents
PLOT_BG   = "#FAFAFC"                           # Very subtle gray for plot areas
GRID_CLR  = "#E0E4E8"
TEXT_CLR  = "#2C3E50"                           # Dark slate for readable text
ACCENT    = "#F39C12"                           # Vibrant orange accent
 
plt.rcParams.update({
    "figure.facecolor"  : BG_COLOR,
    "axes.facecolor"    : PLOT_BG,
    "axes.edgecolor"    : "#BDC3C7",
    "axes.labelcolor"   : TEXT_CLR,
    "axes.titlecolor"   : TEXT_CLR,
    "axes.titleweight"  : "bold",
    "axes.titlesize"    : 14,
    "axes.spines.top"   : False,                 
    "axes.spines.right" : False,
    "xtick.color"       : TEXT_CLR,
    "ytick.color"       : TEXT_CLR,
    "text.color"        : TEXT_CLR,
    "grid.color"        : GRID_CLR,
    "grid.linestyle"    : "--",
    "grid.alpha"        : 0.8,
    "legend.facecolor"  : BG_COLOR,
    "legend.edgecolor"  : "#BDC3C7",
    "legend.fontsize"   : 10,
    "font.family"       : "sans-serif",
    "font.size"         : 11,
})


In [ ]:
# ── 3. REPRODUCIBILITY ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# ── 4. CONFIG ─────────────────────────────────────────────────────────────────
# Make sure your data is structured as data/rice_disease/...
DATA_DIR   = "data/rice_disease" 
OUTPUT_DIR = "outputs_kfold"
os.makedirs(OUTPUT_DIR, exist_ok=True)
CHECKPOINT_DIR = "checkpoints_kfold"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

IMG_SIZE     = 224
BATCH_SIZE   = 16
EPOCHS       = 30
LR           = 1e-4
LR_HEAD      = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = 2
UNFREEZE_EPOCH = 5
N_FOLDS      = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# ── 5. TRANSFORMS ─────────────────────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
 
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 64, IMG_SIZE + 64)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=45),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomPerspective(distortion_scale=0.3, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
 
val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# ── 6. DATASET & KFOLD SETUP ──────────────────────────────────────────────────
full_dataset = datasets.ImageFolder(root=DATA_DIR)
CLASS_NAMES  = full_dataset.classes
NUM_CLASSES  = len(CLASS_NAMES)
print(f"\nClasses ({NUM_CLASSES}) : {CLASS_NAMES}")
print(f"Total images    : {len(full_dataset)}")

all_labels  = [label for _, label in full_dataset.samples]
all_indices = list(range(len(full_dataset)))

class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img, label = self.subset[idx]
        return self.transform(img), label

def plot_class_distribution():
    counts = [all_labels.count(i) for i in range(NUM_CLASSES)]
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(CLASS_NAMES, counts, color=PALETTE, width=0.55,
                  edgecolor="#34495E", linewidth=1.2, alpha=0.9)
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                str(count), ha="center", va="bottom",
                fontsize=12, fontweight="bold", color=TEXT_CLR)
    ax.set_title("Class Distribution in Dataset", pad=15)
    ax.set_ylabel("Number of Images", fontsize=12)
    ax.set_ylim(0, max(counts) * 1.2)
    ax.grid(axis="y", alpha=0.5)
    ax.set_axisbelow(True)
    total = sum(counts)
    pct_labels = [f"{c/total*100:.1f}%" for c in counts]
    ax2 = ax.twinx()
    ax2.set_ylim(0, max(counts) * 1.2)
    ax2.set_yticks([c for c in counts])
    ax2.set_yticklabels(pct_labels, color=ACCENT, fontweight="bold")
    ax2.spines["right"].set_visible(True)
    ax2.spines["right"].set_color(ACCENT)
    ax2.spines["right"].set_linewidth(1.5)
    fig.tight_layout()
    path = os.path.join(OUTPUT_DIR, "01_class_distribution.png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

plot_class_distribution()


In [ ]:
# ── 7. MODEL ──────────────────────────────────────────────────────────────────
def build_resnet50(num_classes, freeze_backbone=True):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(256, num_classes),
    )
    return model

def get_optimizer(model, head_lr=LR_HEAD, backbone_lr=LR):
    head_params     = list(model.fc.parameters())
    backbone_params = [p for p in model.parameters()
                       if not any(p is hp for hp in head_params)]
    return optim.Adam([
        {"params": backbone_params, "lr": backbone_lr},
        {"params": head_params,     "lr": head_lr},
    ], weight_decay=WEIGHT_DECAY)


In [ ]:
# ── 8. TRAINING FUNCTIONS ───────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    loop = tqdm(loader, leave=False, desc="Training")
    for images, labels in loop:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct  += preds.eq(labels).sum().item()
        total    += labels.size(0)
        loop.set_postfix(loss=loss.item())
    return running_loss / total, 100.0 * correct / total
 
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    loop = tqdm(loader, leave=False, desc="Validating")
    with torch.no_grad():
        for images, labels in loop:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct  += preds.eq(labels).sum().item()
            total    += labels.size(0)
    return running_loss / total, 100.0 * correct / total


In [ ]:
# ── 9. K-FOLD TRAINING LOOP ───────────────────────────────────────────────────
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_histories = []
oof_preds = []
oof_labels = []
oof_probs = []

print("\n── 5-Fold Cross Validation Started ──────────────────────────────────────")

for fold, (train_idx, val_idx) in enumerate(skf.split(all_indices, all_labels), start=1):
    print(f"\n{'='*60}")
    print(f"  FOLD {fold}/{N_FOLDS} | Train: {len(train_idx)} | Val: {len(val_idx)}")
    print(f"{'='*60}")
    
    train_subset = Subset(full_dataset, train_idx)
    val_subset   = Subset(full_dataset, val_idx)
    
    train_ds_fold = TransformSubset(train_subset, train_transforms)
    val_ds_fold   = TransformSubset(val_subset, val_transforms)
    
    train_loader = DataLoader(train_ds_fold, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_ds_fold, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
                              
    model = build_resnet50(NUM_CLASSES, freeze_backbone=True).to(DEVICE)
    optimizer = get_optimizer(model)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "lr": []}
    best_val_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    CHECKPOINT = os.path.join(CHECKPOINT_DIR, f"best_resnet50_fold{fold}.pth")
    
    for epoch in range(1, EPOCHS + 1):
        if epoch == UNFREEZE_EPOCH + 1:
            print(f"\n[Epoch {epoch}] Unfreezing backbone for end-to-end fine-tuning ...")
            for param in model.parameters():
                param.requires_grad = True
            optimizer = get_optimizer(model, head_lr=LR_HEAD, backbone_lr=LR)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=EPOCHS - UNFREEZE_EPOCH, eta_min=1e-7)
                
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        scheduler.step()
        
        current_lr = optimizer.param_groups[-1]["lr"]
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["lr"].append(current_lr)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(best_model_wts, CHECKPOINT)
            tag = "  <-- New Best Fold Model Saved!"
        else:
            tag = ""
            
        print(f"Epoch [{epoch:02d}/{EPOCHS}] | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | "
              f"LR: {current_lr:.2e}{tag}")
              
    print(f"\nFold {fold} Best Validation Accuracy : {best_val_acc:.2f}%")
    fold_histories.append(history)
    
    # Store out-of-fold validation predictions using best model
    model.load_state_dict(best_model_wts)
    model.eval()
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            _, preds = outputs.max(1)
            
            oof_preds.extend(preds.cpu().numpy())
            oof_labels.extend(labels.numpy())
            oof_probs.extend(probs.cpu().numpy())


In [ ]:
# ── 10. AGGREGATED EVALUATION & PLOTS ───────────────────────────────────────────
oof_preds  = np.array(oof_preds)
oof_labels = np.array(oof_labels)
oof_probs  = np.array(oof_probs)

acc       = 100.0 * (oof_preds == oof_labels).mean()
precision = precision_score(oof_labels, oof_preds, average="weighted", zero_division=0)
recall    = recall_score(   oof_labels, oof_preds, average="weighted", zero_division=0)
f1        = f1_score(       oof_labels, oof_preds, average="weighted", zero_division=0)

print("\n" + "="*60)
print("  CROSS-VALIDATION AGGREGATED RESULTS (OUT-OF-FOLD)")
print("="*60)
print(f"  Overall Accuracy  : {acc:.2f}%")
print(f"  Overall Precision : {precision:.4f}")
print(f"  Overall Recall    : {recall:.4f}")
print(f"  Overall F1 Score  : {f1:.4f}")
print("\nPer-class Report:")
print(classification_report(oof_labels, oof_preds, target_names=CLASS_NAMES))

# ═════════════════════════════════════════════════════════════════════════════
#  PLOT A — Training Curves (Average across folds)
# ═════════════════════════════════════════════════════════════════════════════
def plot_average_training_curves(histories):
    epochs_range = range(1, EPOCHS + 1)
    
    avg_train_loss = np.mean([h["train_loss"] for h in histories], axis=0)
    avg_val_loss   = np.mean([h["val_loss"] for h in histories], axis=0)
    avg_train_acc  = np.mean([h["train_acc"] for h in histories], axis=0)
    avg_val_acc    = np.mean([h["val_acc"] for h in histories], axis=0)
    lr_history     = histories[0]["lr"] # LR schedule is deterministic
    
    fig = plt.figure(figsize=(18, 5))
    gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.25)
 
    # --- Loss ---
    ax0 = fig.add_subplot(gs[0])
    ax0.plot(epochs_range, avg_train_loss, color=PALETTE[1], lw=2.5, label="Avg Train Loss")
    ax0.plot(epochs_range, avg_val_loss,   color=PALETTE[0], lw=2.5, linestyle="--", label="Avg Val Loss")
    ax0.axvline(UNFREEZE_EPOCH, color=ACCENT, lw=1.5, linestyle=":", label="Unfreeze Backbone")
    ax0.set_title("Average Loss per Epoch")
    ax0.set_xlabel("Epoch"); ax0.set_ylabel("Loss")
    ax0.legend(); ax0.grid(True)
 
    # --- Accuracy ---
    ax1 = fig.add_subplot(gs[1])
    ax1.plot(epochs_range, avg_train_acc, color=PALETTE[1], lw=2.5, label="Avg Train Acc")
    ax1.plot(epochs_range, avg_val_acc,   color=PALETTE[0], lw=2.5, linestyle="--", label="Avg Val Acc")
    ax1.axvline(UNFREEZE_EPOCH, color=ACCENT, lw=1.5, linestyle=":")
    ax1.fill_between(epochs_range, avg_train_acc, avg_val_acc, alpha=0.1, color=PALETTE[1])
    ax1.set_title("Average Accuracy per Epoch")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy (%)")
    ax1.legend(); ax1.grid(True)
 
    # --- Learning Rate ---
    ax2 = fig.add_subplot(gs[2])
    ax2.plot(epochs_range, lr_history, color=ACCENT, lw=2.5)
    ax2.axvline(UNFREEZE_EPOCH, color=TEXT_CLR, lw=1.5, linestyle=":")
    ax2.set_title("Learning Rate Schedule")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Learning Rate")
    ax2.set_yscale("log")
    ax2.grid(True, which="both", linestyle="--", alpha=0.5)
 
    fig.suptitle("ResNet50 5-Fold CV Average Training Summary", fontsize=16, fontweight="bold", y=1.05)
    path = os.path.join(OUTPUT_DIR, "02_avg_training_curves.png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

plot_average_training_curves(fold_histories)

# ═════════════════════════════════════════════════════════════════════════════
#  PLOT B — Confusion Matrix (Out-Of-Fold Aggregated)
# ═════════════════════════════════════════════════════════════════════════════
def plot_confusion_matrix(labels, preds, class_names):
    cm      = confusion_matrix(labels, preds)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
 
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
    fig.suptitle("Confusion Matrix — Out-Of-Fold Performance", fontsize=16, fontweight="bold", y=1.02)
 
    for ax, data, title, fmt in zip(
        axes,
        [cm, cm_norm],
        ["Raw Counts", "Row-Normalised (Recall per Class)"],
        ["d", ".2f"],
    ):
        sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues",
                    xticklabels=class_names, yticklabels=class_names,
                    ax=ax, linewidths=0.5, linecolor=GRID_CLR,
                    annot_kws={"size": 13, "weight": "bold"},
                    cbar_kws={"shrink": 0.85},
                    square=True)
        ax.set_title(title, pad=15)
        ax.set_xlabel("Predicted Label", fontsize=12, fontweight="bold", labelpad=10)
        ax.set_ylabel("True Label",      fontsize=12, fontweight="bold", labelpad=10)
        ax.tick_params(axis="x", rotation=25)
        ax.tick_params(axis="y", rotation=0)
 
    fig.tight_layout()
    path = os.path.join(OUTPUT_DIR, "03_confusion_matrix.png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

plot_confusion_matrix(oof_labels, oof_preds, CLASS_NAMES)

# ═════════════════════════════════════════════════════════════════════════════
#  PLOT C — Per-Class Metrics Bar Chart 
# ═════════════════════════════════════════════════════════════════════════════
def plot_per_class_metrics(labels, preds, class_names):
    report = classification_report(labels, preds,
                                   target_names=class_names,
                                   output_dict=True, zero_division=0)
    metrics    = ["precision", "recall", "f1-score"]
    x          = np.arange(len(class_names))
    bar_width  = 0.22
 
    fig, ax = plt.subplots(figsize=(10, 5.5))
 
    for i, (metric, color) in enumerate(zip(metrics, PALETTE)):
        vals = [report[cls][metric] for cls in class_names]
        bars = ax.bar(x + i * bar_width, vals, bar_width,
                      label=metric.capitalize(), color=color,
                      edgecolor="white", linewidth=1)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.02,
                    f"{val:.2f}", ha="center", fontsize=10,
                    fontweight="bold", color=TEXT_CLR)
 
    ax.set_xticks(x + bar_width)
    ax.set_xticklabels(class_names, fontsize=12)
    ax.set_ylim(0, 1.2)
    ax.set_ylabel("Score", fontsize=12)
    ax.set_title("Per-Class Precision, Recall, and F1-Score (Out-Of-Fold)", pad=15)
    ax.grid(axis="y")
 
    overall_f1 = report["weighted avg"]["f1-score"]
    ax.axhline(overall_f1, color=ACCENT, lw=2, linestyle="--",
               label=f"Weighted F1 = {overall_f1:.2f}")
    
    ax.legend(loc="upper right", frameon=True, shadow=True)
 
    fig.tight_layout()
    path = os.path.join(OUTPUT_DIR, "04_per_class_metrics.png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

plot_per_class_metrics(oof_labels, oof_preds, CLASS_NAMES)

# ═════════════════════════════════════════════════════════════════════════════
#  PLOT D — ROC Curves
# ═════════════════════════════════════════════════════════════════════════════
def plot_roc_curves(labels, probs, class_names):
    y_bin = label_binarize(labels, classes=list(range(len(class_names))))
 
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot([0, 1], [0, 1], color="#7F8C8D", lw=2,
            linestyle="--", label="Random Classifier")
 
    for i, (cls, color) in enumerate(zip(class_names, PALETTE)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
        roc_auc     = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, lw=2.5,
                label=f"{cls} (AUC = {roc_auc:.2f})")
        ax.fill_between(fpr, tpr, alpha=0.05, color=color)
 
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("False Positive Rate", fontsize=12, fontweight="bold")
    ax.set_ylabel("True Positive Rate",  fontsize=12, fontweight="bold")
    ax.set_title("ROC Curves — One-vs-Rest (Out-Of-Fold)", pad=15)
    ax.legend(loc="lower right", frameon=True, shadow=True)
    ax.grid(True)
 
    fig.tight_layout()
    path = os.path.join(OUTPUT_DIR, "05_roc_curves.png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

plot_roc_curves(oof_labels, oof_probs, CLASS_NAMES)

# ═════════════════════════════════════════════════════════════════════════════
#  PLOT E — Summary Scorecard 
# ═════════════════════════════════════════════════════════════════════════════
def plot_scorecard(acc, precision, recall, f1, class_names, labels, preds):
    fig = plt.figure(figsize=(14, 6))
    gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.1, width_ratios=[1, 1.4])
 
    ax_left = fig.add_subplot(gs[0])
    ax_left.axis("off")
    metrics = [
        ("Accuracy",  f"{acc:.2f}%",       PALETTE[1]),
        ("Precision", f"{precision:.4f}",  PALETTE[2]),
        ("Recall",    f"{recall:.4f}",     PALETTE[0]),
        ("F1 Score",  f"{f1:.4f}",         ACCENT),
    ]
    for idx, (name, value, color) in enumerate(metrics):
        y = 0.82 - idx * 0.24
        ax_left.add_patch(mpatches.FancyBboxPatch(
            (0.1, y - 0.08), 0.8, 0.20,
            boxstyle="round,pad=0.03",
            facecolor="#F8F9FA", edgecolor=color, linewidth=2,
            transform=ax_left.transAxes))
        ax_left.text(0.5, y + 0.04, name.upper(), ha="center", va="center",
                     fontsize=12, fontweight="bold", color="#7F8C8D", transform=ax_left.transAxes)
        ax_left.text(0.5, y - 0.04, value, ha="center", va="center",
                     fontsize=24, fontweight="bold", color=color,
                     transform=ax_left.transAxes)
    ax_left.set_title("Overall CV Performance (Out-Of-Fold)", fontsize=15, pad=10)
 
    report  = classification_report(labels, preds, target_names=class_names, output_dict=True, zero_division=0)
    cats    = ["Precision", "Recall", "F1-Score"]
    N       = len(cats)
    angles  = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]
 
    ax_right = fig.add_subplot(gs[1], polar=True)
    ax_right.set_facecolor(PLOT_BG)
    ax_right.set_theta_offset(np.pi / 2)
    ax_right.set_theta_direction(-1)
    ax_right.set_xticks(angles[:-1])
    ax_right.set_xticklabels(cats, fontsize=12, fontweight="bold", color=TEXT_CLR)
    ax_right.set_ylim(0, 1)
    ax_right.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax_right.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], color="#95A5A6", size=9)
 
    for cls, color in zip(class_names, PALETTE):
        vals   = [report[cls]["precision"], report[cls]["recall"], report[cls]["f1-score"]]
        vals  += vals[:1]
        ax_right.plot(angles, vals, color=color, lw=2.5, label=cls)
        ax_right.fill(angles, vals, color=color, alpha=0.15)
 
    ax_right.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), frameon=True, shadow=True)
    ax_right.set_title("Per-Class Radar Breakdown", fontsize=15, pad=20)
 
    fig.suptitle("CV Model Performance Scorecard", fontsize=18, fontweight="bold", y=1.05)
    path = os.path.join(OUTPUT_DIR, "06_scorecard.png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
 
plot_scorecard(acc, precision, recall, f1, CLASS_NAMES, oof_labels, oof_preds)
print("\n" + "="*60)
print("  All outputs and models successfully saved!")
print("="*60)
